<a href="https://colab.research.google.com/github/ArafathUIU/AskYourDocs/blob/main/AskYourDocs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install Dependencies


In [1]:
!pip install sentence-transformers faiss-cpu PyMuPDF python-docx openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.8 MB/s eta 0:00:00


Step 2: File Upload

In [8]:
from google.colab import files
uploaded = files.upload()  # Upload your PDFs / DOCX files

Saving Introduction-to-AI-and-Basic-Concepts.pdf to Introduction-to-AI-and-Basic-Concepts.pdf


Step 3: Load & Parse Documents

In [9]:
import fitz  # PyMuPDF
from docx import Document

def load_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def load_docx(file_path):
    doc = Document(file_path)
    return "\n".join([p.text for p in doc.paragraphs])

documents = []
for file_name in uploaded.keys():
    if file_name.endswith(".pdf"):
        documents.append(load_pdf(file_name))
    elif file_name.endswith(".docx"):
        documents.append(load_docx(file_name))

Step 4: Chunking

In [10]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))

Step 5: Generate Embeddings (SentenceTransformers)

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Generate embeddings
embeddings = model.encode(all_chunks, convert_to_numpy=True)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"Total chunks in index: {index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks in index: 7


Step 6: Query & Retrieve Top-k Chunks

In [12]:
def retrieve(query, top_k=3):
    q_emb = model.encode([query])
    distances, indices = index.search(q_emb, top_k)
    results = [all_chunks[i] for i in indices[0]]
    return results

query = "Your question here"
retrieved_chunks = retrieve(query)
print("Retrieved Chunks:")
for c in retrieved_chunks:
    print(c[:200], "...")  # show first 200 chars

Retrieved Chunks:
72 Position Goalkeeper Right Back Center Back Midfielder Left Winger Striker Midfielder Right Winger Center Back Left Winger Forward Feature Vector Feature Label 𝑥𝑖= ℎ𝑒𝑖𝑔ℎ𝑡, 𝑤𝑒𝑖𝑔ℎ𝑡, 𝑖= 1, 2, . . . , 1 ...
Insurance Forms Date of submission, who filled it out, who wrote it, and the type of request Content of open-ended questions Podcasts Date published, host name, and podcast category Audio recording an ...
Introduction to Artificial Intelligence and Fundamental Concepts Dr. Melike PALSÜ KURT What Will We Learn? • What is Artificial Intelligence? • Can Machines Think, and How Can They Think? • Turing Tes ...


In [19]:
# ------------------------------
# Step 7: Gemini API - Document Summary + Question Answer
# ------------------------------

# Flatten retrieved chunks into readable context
context = "\n\n".join(retrieved_chunks)

# Your actual question about the document summary
query = "Can you summarize the key points of this document?"

# Build a clear prompt instructing Gemini to summarize
prompt = f"""
You are a helpful assistant. Read the following document and tell us about The Intelligence Revolution:
The Birth of Artificial Intelligence in 20 words.
Use only the information provided in the text.

Document Text:
{context}

Question:
{query}

Answer:
"""

# Import Gemini client
from google import genai
import os

# Set your API key (keep it secret!)
os.environ["GEMINI_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # Replace with your key

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Call Gemini API to generate the answer
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# Display the summary answer
print("Document Summary / Answer:")
print(response.text)

Document Summary / Answer:
John McCarthy's 1956 proposal: intelligence can be precisely described for machine simulation, sparking the birth of artificial intelligence.
